In [16]:
from app_motion.prepare_dataset import MOTReIDMotionDataset
from app_motion.sampler import RandomIdentitySampler
from torch.utils.data import DataLoader
import torch

In [2]:
train_dataset = MOTReIDMotionDataset(
    annotation_file='/media/hung/Work/Project/AI Engineer/CPV/bytetrack-LSTM/datasets/mot/annotations/train_half.json',
    img_dir='/media/hung/Work/Project/AI Engineer/CPV/bytetrack-LSTM/datasets/mot/train',
    K=5)

loading annotations into memory...
Done (t=0.17s)
creating index...
index created!


In [3]:
unique_ids = list(train_dataset.tracks.keys())
num_classes = len(unique_ids)
id_to_class = {track_id: i for i, track_id in enumerate(unique_ids)}

In [4]:
train_dataset.samples[2]['track_id']

1

In [5]:
sampler = RandomIdentitySampler(
        train_dataset,
        batch_size=32,
        num_instances=4
    )

In [10]:
indices = list(iter(sampler))

print("Num indices:", len(indices))
print(indices[:64])

Num indices: 56320
[37333, 37324, 37332, 37318, 43964, 43919, 44020, 43991, 18063, 17900, 17800, 17789, 55598, 55629, 55641, 55567, 42491, 42469, 42492, 42481, 27953, 27930, 27580, 27608, 49612, 49598, 49602, 49601, 43745, 43686, 43704, 43737, 56734, 56726, 56729, 56738, 35400, 35314, 35419, 35272, 8217, 8231, 8243, 8242, 50613, 50720, 50733, 50684, 1583, 1715, 1802, 1859, 55288, 55307, 55322, 55314, 45798, 45710, 45665, 45773, 34943, 34942, 34939, 34941]


In [11]:
indices = list(iter(sampler))

batch_size = 32

for start in range(0, batch_size, 4):

    idxs = indices[start:start+4]

    pids = [
        train_dataset.samples[i]["track_id"]
        for i in idxs
    ]

    print(idxs)
    print(pids)
    print()

[43844, 43857, 43803, 43802]
[231, 231, 231, 231]

[53452, 53526, 53424, 53428]
[308, 308, 308, 308]

[56105, 56096, 56086, 56115]
[345, 345, 345, 345]

[53922, 53906, 53920, 53842]
[313, 313, 313, 313]

[39170, 39143, 39269, 39241]
[201, 201, 201, 201]

[48266, 48287, 48281, 48245]
[264, 264, 264, 264]

[33684, 33679, 33680, 33668]
[139, 139, 139, 139]

[1352, 1363, 1338, 1297]
[7, 7, 7, 7]



In [12]:
loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

batch = next(iter(loader))

imgs, motions, track_ids = batch

print(track_ids)

tensor([281, 281, 281, 281, 259, 259, 259, 259,  91,  91,  91,  91, 134, 134,
        134, 134, 175, 175, 175, 175, 273, 273, 273, 273,  56,  56,  56,  56,
        138, 138, 138, 138])


In [14]:
def debug_sampler(dataset, sampler):

    idxs = list(iter(sampler))[:32]

    pids = [
        dataset.samples[i]["track_id"]
        for i in idxs
    ]

    print("Indices:")
    print(idxs)

    print("\nPIDs:")
    print(pids)

    print("\nGroups:")

    for i in range(0, len(pids), sampler.num_instances):

        group = pids[
            i:i+sampler.num_instances
        ]

        print(group)

debug_sampler(
    train_dataset,
    sampler
)

Indices:
[36412, 36348, 36449, 36367, 38231, 38228, 38220, 38233, 7443, 7534, 7590, 7519, 33277, 33260, 33247, 33292, 56163, 56175, 56132, 56125, 34945, 34951, 34954, 34952, 48694, 48697, 48696, 48692, 387, 376, 384, 357]

PIDs:
[184, 184, 184, 184, 195, 195, 195, 195, 37, 37, 37, 37, 124, 124, 124, 124, 346, 346, 346, 346, 170, 170, 170, 170, 273, 273, 273, 273, 3, 3, 3, 3]

Groups:
[184, 184, 184, 184]
[195, 195, 195, 195]
[37, 37, 37, 37]
[124, 124, 124, 124]
[346, 346, 346, 346]
[170, 170, 170, 170]
[273, 273, 273, 273]
[3, 3, 3, 3]


In [17]:
labels = track_ids

for pid in torch.unique(labels):

    print(
        pid.item(),
        (labels == pid).sum().item()
    )

56 4
91 4
134 4
138 4
175 4
259 4
273 4
281 4
